# Principio de Segregación de Interfaces (ISP)

Ningún cliente debe depender de métodos que no necesita.

## Sin aplicar ISP

Una interfaz gorda obliga al restaurante y al repartidor a implementar operaciones irrelevantes.

In [ ]:
from abc import ABC, abstractmethod

class ParticipantePlataforma(ABC):
    def __init__(self, nombre, identificador): self.nombre = nombre; self.identificador = identificador
    @abstractmethod
    def preparar(self, pedido): pass
    @abstractmethod
    def entregar(self, pedido): pass
    @abstractmethod
    def cobrar(self, pedido): pass

class RestauranteForzado(ParticipantePlataforma):
    def __init__(self, nombre, identificador): super().__init__(nombre, identificador); self.preparados = 0
    def preparar(self, pedido): self.preparados += 1; return f'{self.nombre} preparó {pedido}'
    def entregar(self, pedido): raise NotImplementedError('El restaurante no entrega')
    def cobrar(self, pedido): return f'Cobro registrado para {pedido}'

class RepartidorForzado(ParticipantePlataforma):
    def __init__(self, nombre, identificador): super().__init__(nombre, identificador); self.entregas = 0
    def preparar(self, pedido): raise NotImplementedError('El repartidor no cocina')
    def entregar(self, pedido): self.entregas += 1; return f'{self.nombre} entregó {pedido}'
    def cobrar(self, pedido): raise NotImplementedError('El repartidor no cobra')

r = RestauranteForzado('Sazón Caribe','R-01'); d = RepartidorForzado('Luis','D-01')
print(r.preparar('Pedido 10'))
for accion in (r.entregar, d.preparar):
    try: print(accion('Pedido 10'))
    except NotImplementedError as error: print('Método irrelevante impuesto:', error)

## Aplicando ISP

Las capacidades se dividen. Cada implementación y cliente conocen únicamente las operaciones que utilizan.

In [ ]:
class Preparador(ABC):
    def __init__(self, nombre, zona): self.nombre = nombre; self.zona = zona
    @abstractmethod
    def aceptar(self, pedido): pass
    @abstractmethod
    def preparar(self, pedido): pass

class Transportador(ABC):
    def __init__(self, nombre, vehiculo): self.nombre = nombre; self.vehiculo = vehiculo
    @abstractmethod
    def recoger(self, pedido): pass
    @abstractmethod
    def entregar(self, pedido): pass

class Restaurante(Preparador):
    def __init__(self, nombre, zona): super().__init__(nombre, zona); self.cola = []
    def aceptar(self, pedido): self.cola.append(pedido); return f'{pedido} aceptado'
    def preparar(self, pedido): return f'{self.nombre} preparó {pedido} en {self.zona}'

class Repartidor(Transportador):
    def __init__(self, nombre, vehiculo): super().__init__(nombre, vehiculo); self.pedido_actual = None
    def recoger(self, pedido): self.pedido_actual = pedido; return f'{self.nombre} recogió {pedido}'
    def entregar(self, pedido): self.pedido_actual = None; return f'{self.nombre} entregó {pedido} en {self.vehiculo}'

class CocinaCliente:
    def __init__(self, preparador, pedido): self.preparador = preparador; self.pedido = pedido
    def solicitar(self): return self.preparador.aceptar(self.pedido)
    def obtener(self): return self.preparador.preparar(self.pedido)

class EntregaCliente:
    def __init__(self, transportador, pedido): self.transportador = transportador; self.pedido = pedido
    def iniciar(self): return self.transportador.recoger(self.pedido)
    def finalizar(self): return self.transportador.entregar(self.pedido)

cocina = CocinaCliente(Restaurante('Sazón Caribe','Centro'),'Pedido 20')
entrega = EntregaCliente(Repartidor('Luis','bicicleta'),'Pedido 20')
print(cocina.solicitar()); print(cocina.obtener()); print(entrega.iniciar()); print(entrega.finalizar())

`CocinaCliente` depende solo de preparación y `EntregaCliente` solo de transporte. Ninguna clase implementa métodos inútiles.